# HydraShield Protection Optimisation

This notebook demonstrates the HydraShield hydration control pipeline: computing protection zones, optimising water allocation, and planning interventions.

In [ ]:
import sys
import os
import numpy as np

sys.path.insert(0, os.path.abspath('..'))

from src.gis_mapping.mapping import ProtectionZoneMapper
from src.hydration_control.water_optimiser import WaterOptimiser
from src.hydration_control.intervention import InterventionPlanner

## 1. Build Protection Zones

Compute Critical Protection Zones around vulnerable assets (schools, hospitals, evacuation routes).

In [ ]:
mapper = ProtectionZoneMapper()

assets = [
    {'id': 'school', 'type': 'school', 'centroid': (10.0, 20.0)},
    {'id': 'hospital', 'type': 'hospital', 'centroid': (11.0, 21.0)},
    {'id': 'evac_route', 'type': 'evacuation_route', 'centroid': (10.5, 20.5)},
]

zones = mapper.build_protection_zones(
    assets, ros_m_per_min=5.0, probability_of_spread=0.4, lead_time_min=60.0
)

for z in zones:
    print(f"{z.asset_id}: radius={z.radius_m:.0f}m, area={z.area_m2:.0f}m2, risk={z.risk_level}")

## 2. Water Allocation (Water-Scarce Mode)

Allocate limited water across zones by priority.

In [ ]:
optimiser = WaterOptimiser(water_available_m3=500.0)

priorities = [3.0, 5.0, 4.0]  # hospital highest
areas = [z.area_m2 for z in zones]

allocations = optimiser.allocate_water(priorities, areas)

for z, alloc in zip(zones, allocations):
    print(f"{z.asset_id}: allocated {alloc:.1f} m3")

## 3. Intervention Planning

Build intervention plans with traffic-light recommendations.

In [ ]:
planner = InterventionPlanner()

plans = planner.build_plan(
    zone_ids=[z.asset_id for z in zones],
    water_volumes_m3=allocations,
    confidences=[0.9, 0.95, 0.6],
)

for p in plans:
    print(f"{p.zone_id}: {p.recommendation.upper()} | {p.water_volume_m3:.1f} m3 | start={p.start_time_h:.1f}h | dur={p.duration_h:.1f}h")

## 4. Water-Use Efficiency

Quantify risk reduction per cubic metre of water.

In [ ]:
wuer = optimiser.compute_wuer(
    risk_baseline=0.8, risk_hydrashield=0.3, water_volume_m3=sum(allocations)
)
print(f"WUER: {wuer.wuer:.4f} risk-reduction per m3")
print(f"Water savings vs conventional: {optimiser.water_savings(2000.0, sum(allocations)):.1f}%")